# Fase 1: EDA y Baseline

En este notebook realizaremos la descarga de los datos de BreastMNIST y configuraremos el pipeline inicial de TensorFlow.

In [ ]:
import os
from medmnist import BreastMNIST

# Definimos la ruta apuntando a la carpeta de datos crudos creada previamente
base_dir = os.path.join('..', 'data', 'raw', 'breastmnist')
splits = ['train', 'val', 'test']

# Crear estructura de carpetas por clase (0: maligno, 1: benigno/normal)
for split in splits:
    for class_id in ['0', '1']:
        os.makedirs(os.path.join(base_dir, split, class_id), exist_ok=True)

def save_medmnist_to_disk(split_name):
    print(f"Procesando partición: {split_name}...")
    # Se exige usar la resolución nativa de 28x28 para el baseline
    dataset = BreastMNIST(split=split_name, download=True, size=28)
    
    for i in range(len(dataset)):
        img, label = dataset[i]
        class_label = str(label[0])
        
        img_path = os.path.join(base_dir, split_name, class_label, f"{split_name}_{i}.jpg")
        img.save(img_path)

# Ejecutar para las 3 particiones
# save_medmnist_to_disk('train') # Ejecutado por CLI
# save_medmnist_to_disk('val')
# save_medmnist_to_disk('test')
print("¡Imágenes listas en data/raw!")

In [ ]:
import tensorflow as tf
import os

BATCH_SIZE = 32
IMG_SIZE = (28, 28) # Resolución obligatoria para la Fase 1
base_dir = os.path.join('..', 'data', 'raw', 'breastmnist')

def create_tf_dataset(split_name):
    dir_path = os.path.join(base_dir, split_name)
    dataset = tf.keras.utils.image_dataset_from_directory(
        dir_path,
        shuffle=(split_name == 'train'), # Solo barajamos el entrenamiento
        batch_size=BATCH_SIZE,
        image_size=IMG_SIZE,
        color_mode='rgb' # Requerido para modelos de Transfer Learning (3 canales)
    )
    # Optimización de rendimiento
    AUTOTUNE = tf.data.AUTOTUNE
    return dataset.cache().prefetch(buffer_size=AUTOTUNE)

# Instanciamos los datasets
train_dataset = create_tf_dataset('train')
val_dataset = create_tf_dataset('val')
test_dataset = create_tf_dataset('test')

print("¡Datasets de TensorFlow configurados y optimizados!")

# Fase 2: Estudio Comparativo Avanzado
En esta fase implementamos técnicas de regularización, una segunda arquitectura CNN (EfficientNetB0) y un Vision Transformer (ViT).

## 1. Regularización y Mecanismos de Control

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

# 1. Capa de Data Augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
])

# 2. Callback de Early Stopping
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

## 2. Pipeline Multirresolución (128x128)

In [ ]:
# Configuración del Pipeline para la Fase 2 (128x128)
IMG_SIZE_128 = (128, 128)
BATCH_SIZE = 32
base_dir = os.path.join('..', 'data', 'raw', 'breastmnist')

def create_tf_dataset_128(split_name):
    dir_path = os.path.join(base_dir, split_name)
    dataset = tf.keras.utils.image_dataset_from_directory(
        dir_path,
        shuffle=(split_name == 'train'),
        batch_size=BATCH_SIZE,
        image_size=IMG_SIZE_128,
        color_mode='rgb'
    )
    return dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

# Datasets de alta resolución
train_dataset_128 = create_tf_dataset_128('train')
val_dataset_128 = create_tf_dataset_128('val')
test_dataset_128 = create_tf_dataset_128('test')

## 3. Segunda Arquitectura CNN (EfficientNetB0)

In [ ]:
def build_efficientnet_baseline(input_shape):
    base_model_2 = tf.keras.applications.EfficientNetB0(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base_model_2.trainable = False

    inputs = tf.keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = base_model_2(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy'),
                 tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return model, base_model_2

# Instanciación para resolución 128x128
cnn_model_128, cnn_base_128 = build_efficientnet_baseline((128, 128, 3))

## 4. Vision Transformer (ViT) Adaptado

In [ ]:
class PatchExtract(layers.Layer):
    def __init__(self, patch_size):
        super(PatchExtract, self).__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID',
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

class PatchEmbedding(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super(PatchEmbedding, self).__init__()
        self.num_patches = num_patches
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patch) + self.position_embedding(positions)

def build_vit_classifier(image_size, patch_size, num_patches, projection_dim, num_heads, transformer_layers, mlp_head_units):
    inputs = layers.Input(shape=(image_size, image_size, 3))
    augmented = data_augmentation(inputs)
    patches = PatchExtract(patch_size)(augmented)
    embeddings = PatchEmbedding(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization(epsilon=1e-6)(embeddings)
        attention_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim, dropout=0.1)(x1, x1)
        x2 = layers.Add()([attention_output, embeddings])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = layers.Dense(projection_dim * 2, activation=tf.nn.gelu)(x3)
        x3 = layers.Dropout(0.1)(x3)
        x3 = layers.Dense(projection_dim, activation=tf.nn.gelu)(x3)
        x3 = layers.Dropout(0.1)(x3)
        embeddings = layers.Add()([x3, x2])

    representation = layers.LayerNormalization(epsilon=1e-6)(embeddings)
    representation = layers.GlobalAveragePooling1D()(representation)
    representation = layers.Dropout(0.3)(representation)

    for units in mlp_head_units:
        representation = layers.Dense(units, activation=tf.nn.gelu)(representation)
        representation = layers.Dropout(0.2)(representation)

    outputs = layers.Dense(1, activation='sigmoid')(representation)
    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy'),
                 tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return model

# Instanciación
vit_model_28 = build_vit_classifier(
    image_size=28, patch_size=4, num_patches=49, 
    projection_dim=64, num_heads=4, transformer_layers=4, mlp_head_units=[128, 64]
)

vit_model_128 = build_vit_classifier(
    image_size=128, patch_size=16, num_patches=64, 
    projection_dim=128, num_heads=4, transformer_layers=6, mlp_head_units=[256, 128]
)